In [49]:
from transformers import ViTForImageClassification, ViTImageProcessor
from PIL import Image
import kagglehub
import requests
import os
import torch
import random
import numpy as np
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim



In [ ]:
processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224')
optimizer = optim.SGD(params=)

path = kagglehub.dataset_download("dansbecker/food-101")



SyntaxError: invalid syntax (3474842701.py, line 3)

In [ ]:
print(path)
print(os.listdir(os.path.join(path, 'food-101', 'food-101')))
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

C:\Users\villa\.cache\kagglehub\datasets\dansbecker\food-101\versions\1
['.DS_Store', 'images', 'license_agreement.txt', 'meta', 'README.txt']
PyTorch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 3060 Ti


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Random seed set to: {SEED}")

cuda
Random seed set to: 42


In [ ]:
ROOT_PATH = os.path.join(path, 'food-101', 'food-101', 'images')
classes_path = os.path.join(path, 'food-101', 'food-101', 'meta', 'classes.txt')
train_image_paths = os.path.join(path, 'food-101', 'food-101', 'meta', 'train.txt')
test_image_paths = os.path.join(path, 'food-101', 'food-101', 'meta', 'test.txt')

with open(classes_path) as f:
	classes = f.read().splitlines()

class_to_index = {}

for index, class_name in enumerate(classes):
	class_to_index[class_name] = index

# print(list(class_to_index.items()))


with open(test_image_paths) as f:
	lines = f.read().splitlines()

test_images_paths = []
test_labels = []

for line in lines:
	parts = line.split('/')
	test_images_paths.append(os.path.join(ROOT_PATH, line + ".jpg"))
	test_labels.append(class_to_index[parts[0]])

with open(train_image_paths) as f:
	lines = f.read().splitlines()

train_images_paths = []
train_labels = []

for line in lines:
	parts = line.split('/')
	train_images_paths.append(os.path.join(ROOT_PATH, line + ".jpg"))
	train_labels.append(class_to_index[parts[0]])

# print (test_images_paths[0])
# print (test_labels)

In [ ]:
class Image_Dataset(Dataset):
	def __init__(self, image_paths, labels, processor):
		self.image_paths = image_paths
		self.labels = labels
		self.processor = processor

	def __getitem__(self, index):
		path = self.image_paths[index]
		image = Image.open(path).convert("RGB")
		inputs = self.processor(images = image, return_tensors = "pt")
		pixel_values = inputs["pixel_values"].squeeze(0)
		return pixel_values, self.labels[index]

	def __len__(self):
		return len(self.image_paths)




In [ ]:
testing_Dataset = Image_Dataset(test_images_paths, test_labels, processor)
testing_Dataloader = DataLoader(testing_Dataset, batch_size=32)

training_Dataset = Image_Dataset(train_images_paths, train_labels, processor)
training_Dataloader = DataLoader(training_Dataset, batch_size=32, shuffle=True)





First Stage Training:
- Only Final Layer is unfrozen to train only the classification head. 

In [ ]:
num_epochs = 5

for epochs in num_epochs:
	